# Probability Concepts in Python
### CFA Level 1 Quantitative Methods | Applied to Real Market Scenarios

---

## The Scenario

You are a quantitative analyst at a macro hedge fund.

The Federal Reserve is meeting next week.
Your portfolio manager walks in and asks:

> "Given what we know about inflation and employment data,
> what is the probability the Fed cuts rates?
> And if they do — what happens to our equity and bond positions?"

This is not a textbook problem.
This is Tuesday morning at 8am.

To answer it rigorously, you need probability theory —
the same theory the CFA curriculum teaches,
applied the way practitioners actually use it.

**What this notebook builds:**
- A framework for assigning and updating probabilities to macro scenarios
- Expected return calculations across scenarios
- Portfolio variance from first principles
- Bayes' theorem to update your view when new data arrives
- Real market data to validate every concept

**Data sources:**
- Price and macro data — Financial Modeling Prep (FMP) API
- Economic data — FRED API

## Setup

We will use the following libraries:

- `requests` — to pull data from FMP and FRED APIs
- `pandas` — for data manipulation
- `numpy` — for mathematical operations
- `matplotlib` — for visualizations
- `scipy` — for statistical functions

We will use two data sources throughout this notebook:
- **FMP API** — real market price data
- **FRED API** — Federal Reserve economic data (Fed Funds Rate, CPI)

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as stats
import warnings
from dotenv import load_dotenv
import os

load_dotenv()
warnings.filterwarnings('ignore')

FMP_KEY  = os.getenv("FMP_KEY")
FRED_KEY = os.getenv("FRED_KEY")

# Plot styling
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['axes.facecolor']   = '#161B22'
plt.rcParams['axes.edgecolor']   = '#30363D'
plt.rcParams['text.color']       = '#E6EDF3'
plt.rcParams['axes.labelcolor']  = '#E6EDF3'
plt.rcParams['xtick.color']      = '#8B949E'
plt.rcParams['ytick.color']      = '#8B949E'
plt.rcParams['grid.color']       = '#21262D'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

print("Libraries loaded successfully")
print("FMP Key loaded  ✓" if FMP_KEY  else "FMP Key missing ✗")
print("FRED Key loaded ✓" if FRED_KEY else "FRED Key missing ✗")

Libraries loaded successfully
FMP Key loaded  ✓
FRED Key loaded ✓


## 1. Building a Probability Framework for Fed Scenarios

Before we touch any data, we need to think like a probability theorist.

The Fed has three options next week:
- **Cut rates** — bullish for equities, bullish for bonds
- **Hold rates** — neutral
- **Hike rates** — bearish for equities, bearish for bonds

These are **mutually exclusive** (only one can happen) and
**exhaustive** (one of them must happen).

This means their probabilities must sum to 1:

$$P(\text{Cut}) + P(\text{Hold}) + P(\text{Hike}) = 1$$

This is the **addition rule for exhaustive events** — CFA Reading 4.

Where do the probabilities come from?

In practice, analysts use three sources:
- **Fed Funds Futures** — market-implied probabilities
- **Empirical frequency** — how often has the Fed cut/held/hiked
  given similar macro conditions historically?
- **Subjective judgment** — your PM's view based on Fed communication

We will use all three and show how they differ.

In [2]:
# =============================================================================
# CELL 5 — Fed Decision History: Empirical Probabilities from FRED
# =============================================================================
# We pull the Fed Funds Rate from FRED and classify each monthly
# change as a CUT, HOLD, or HIKE.
# This gives us the historical base rate for each decision.
# =============================================================================

def get_fred_series(series_id, api_key, from_date="2000-01-01", to_date="2024-01-01"):
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id, "api_key": api_key,
        "file_type": "json", "observation_start": from_date,
        "observation_end": to_date,
    }
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame(data["observations"])
    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.set_index("date").dropna()
    return df["value"]

# Pull Fed Funds Rate
print("Pulling Fed Funds Rate from FRED...")
fed_funds = get_fred_series("FEDFUNDS", FRED_KEY)
print(f"  ✓ {len(fed_funds)} monthly observations loaded")

# ── CLASSIFY EACH MONTH AS CUT / HOLD / HIKE ─────────────────────────────────
changes = fed_funds.diff().dropna()

def classify(change):
    if change < -0.01:  return "CUT"
    elif change > 0.01: return "HIKE"
    else:               return "HOLD"

decisions = changes.apply(classify)

# ── EMPIRICAL PROBABILITIES ───────────────────────────────────────────────────
counts = decisions.value_counts()
total  = len(decisions)

p_empirical = {
    "CUT" : counts.get("CUT",  0) / total,
    "HOLD": counts.get("HOLD", 0) / total,
    "HIKE": counts.get("HIKE", 0) / total,
}

# ── MARKET-IMPLIED (hypothetical — as if reading Fed Funds Futures) ───────────
# In practice you would scrape CME FedWatch tool
# We use a realistic hypothetical for the scenario
p_market = {"CUT": 0.35, "HOLD": 0.55, "HIKE": 0.10}

# ── SUBJECTIVE (your PM's view) ───────────────────────────────────────────────
p_subjective = {"CUT": 0.50, "HOLD": 0.40, "HIKE": 0.10}

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FED DECISION PROBABILITIES — THREE SOURCES")
print(f"{'='*60}")
print(f"\n  {'Scenario':<8} {'Empirical':>12} {'Market-Implied':>16} {'PM View':>10}")
print(f"  {'-'*50}")

for scenario in ["CUT", "HOLD", "HIKE"]:
    print(f"  {scenario:<8} {p_empirical[scenario]:>12.2%} "
          f"{p_market[scenario]:>16.2%} {p_subjective[scenario]:>10.2%}")

print(f"  {'-'*50}")
print(f"  {'SUM':<8} {sum(p_empirical.values()):>12.2%} "
      f"{sum(p_market.values()):>16.2%} {sum(p_subjective.values()):>10.2%}")

print(f"\n  Historical record (2000–2023):")
for scenario in ["CUT", "HOLD", "HIKE"]:
    print(f"  {scenario}: {counts.get(scenario, 0)} months out of {total}")

print(f"\n  The market is pricing a higher probability of a cut")
print(f"  than history suggests. Your PM is even more dovish.")
print(f"  These disagreements are where alpha is generated.")
print(f"{'='*60}")

Pulling Fed Funds Rate from FRED...
  ✓ 289 monthly observations loaded

FED DECISION PROBABILITIES — THREE SOURCES

  Scenario    Empirical   Market-Implied    PM View
  --------------------------------------------------
  CUT            25.69%           35.00%     50.00%
  HOLD           34.38%           55.00%     40.00%
  HIKE           39.93%           10.00%     10.00%
  --------------------------------------------------
  SUM           100.00%          100.00%    100.00%

  Historical record (2000–2023):
  CUT: 74 months out of 288
  HOLD: 99 months out of 288
  HIKE: 115 months out of 288

  The market is pricing a higher probability of a cut
  than history suggests. Your PM is even more dovish.
  These disagreements are where alpha is generated.


## 2. Expected Return Under Macro Scenarios

Now the PM asks the real question:

> "Given these probabilities — what is our portfolio's expected return?"

This is the **expected value** formula from CFA Reading 4:

$$E(R) = \sum_{i} P(\text{scenario}_i) \times R_i$$

We have two positions:
- **SPY** — S&P 500 ETF (equity)
- **TLT** — 20+ Year Treasury Bond ETF (bonds)

For each Fed scenario we assign an expected return
based on historical behavior of these assets
during cutting, holding, and hiking cycles.

Then we compute the portfolio expected return
as the probability-weighted average across all scenarios.

This is exactly what a macro analyst does before every Fed meeting.

In [4]:
# =============================================================================
# CELL 6 — Expected Return Under Fed Scenarios
# =============================================================================
# We pull historical SPY and TLT returns and compute their average
# performance during Fed cutting, holding, and hiking cycles.
# These become our scenario returns — grounded in real data.
# =============================================================================

def get_prices(ticker, api_key, from_date="2000-01-01", to_date="2024-01-01"):
    url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
    params = {"symbol": ticker, "from": from_date, "to": to_date, "apikey": api_key}
    response = requests.get(url, params=params)
    data = response.json()
    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
    elif isinstance(data, dict) and "historical" in data:
        df = pd.DataFrame(data["historical"])
    else:
        return None
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# Pull SPY and TLT
print("Pulling price data from FMP...")
spy = get_prices("SPY", FMP_KEY)
tlt = get_prices("TLT", FMP_KEY)
print(f"  ✓ SPY — {len(spy)} days")
print(f"  ✓ TLT — {len(tlt)} days")

# Monthly returns
spy_monthly = spy.resample("ME").last().pct_change().dropna()
tlt_monthly = tlt.resample("ME").last().pct_change().dropna()

# Align Fed decisions index to month-end to match price data
decisions_monthly = decisions.copy()
decisions_monthly.index = decisions_monthly.index.to_period("M").to_timestamp("M")

common_idx  = spy_monthly.index.intersection(tlt_monthly.index).intersection(decisions_monthly.index)
spy_m       = spy_monthly[common_idx]
tlt_m       = tlt_monthly[common_idx]
decisions_m = decisions_monthly[common_idx]

# ── AVERAGE RETURNS BY FED REGIME ────────────────────────────────────────────
scenario_returns = {}
for scenario in ["CUT", "HOLD", "HIKE"]:
    mask = decisions_m == scenario
    scenario_returns[scenario] = {
        "SPY": spy_m[mask].mean(),
        "TLT": tlt_m[mask].mean(),
        "n"  : mask.sum()
    }

# ── PORTFOLIO WEIGHTS ─────────────────────────────────────────────────────────
w_spy = 0.60
w_tlt = 0.40

# ── EXPECTED RETURN CALCULATION ───────────────────────────────────────────────
print(f"\n{'='*65}")
print("EXPECTED RETURN UNDER FED SCENARIOS")
print(f"{'='*65}")
print(f"\n  Portfolio: {w_spy:.0%} SPY + {w_tlt:.0%} TLT")

print(f"\n  {'Scenario':<8} {'P(Market)':>10} {'SPY Ret':>9} {'TLT Ret':>9} "
      f"{'Port Ret':>10} {'Weighted':>10} {'Obs':>5}")
print(f"  {'-'*62}")

e_return_market = 0
e_return_pm     = 0

for scenario in ["CUT", "HOLD", "HIKE"]:
    p_m    = p_market[scenario]
    p_pm   = p_subjective[scenario]
    r_spy  = scenario_returns[scenario]["SPY"]
    r_tlt  = scenario_returns[scenario]["TLT"]
    r_port = w_spy * r_spy + w_tlt * r_tlt
    n      = scenario_returns[scenario]["n"]

    e_return_market += p_m  * r_port
    e_return_pm     += p_pm * r_port

    print(f"  {scenario:<8} {p_m:>10.2%} {r_spy:>9.2%} {r_tlt:>9.2%} "
          f"{r_port:>10.2%} {p_m*r_port:>10.4%} {n:>5}")

print(f"  {'-'*62}")
print(f"  {'E(R) — Market probabilities':>40}  {e_return_market:>10.4%}")
print(f"  {'E(R) — PM probabilities':>40}  {e_return_pm:>10.4%}")

print(f"""
  INTERPRETATION
  Using market-implied probabilities: E(R) = {e_return_market:.2%} per month
  Using your PM's probabilities:      E(R) = {e_return_pm:.2%} per month

  The PM is more bullish than the market on cuts ({p_subjective['CUT']:.0%} vs {p_market['CUT']:.0%}).
  This translates to a {'higher' if e_return_pm > e_return_market else 'lower'} expected return.
  The difference ({abs(e_return_pm - e_return_market):.2%}/month) is the PM's active bet.
""")
print(f"{'='*65}")

Pulling price data from FMP...
  ✓ SPY — 5000 days
  ✓ TLT — 5000 days

EXPECTED RETURN UNDER FED SCENARIOS

  Portfolio: 60% SPY + 40% TLT

  Scenario  P(Market)   SPY Ret   TLT Ret   Port Ret   Weighted   Obs
  --------------------------------------------------------------
  CUT          35.00%    -0.00%     0.93%      0.37%    0.1306%    49
  HOLD         55.00%     1.66%    -0.36%      0.85%    0.4675%    89
  HIKE         10.00%     0.17%     0.16%      0.17%    0.0167%   100
  --------------------------------------------------------------
               E(R) — Market probabilities     0.6148%
                   E(R) — PM probabilities     0.5432%

  INTERPRETATION
  Using market-implied probabilities: E(R) = 0.61% per month
  Using your PM's probabilities:      E(R) = 0.54% per month

  The PM is more bullish than the market on cuts (50% vs 35%).
  This translates to a lower expected return.
  The difference (0.07%/month) is the PM's active bet.



## 3. Conditional Probability — Reading the Data Correctly

The PM pushes back:

> "Those cut-cycle SPY returns look wrong. 
> Aren't we more likely to cut when inflation is already low?
> Shouldn't we condition on the macro environment?"

This is exactly right. And it leads us to **conditional probability**.

$$P(A | B) = \frac{P(A \cap B)}{P(B)}$$

The unconditional probability of a cut is 25.69%.

But the probability of a cut **given that inflation is above 4%**
is very different from the probability of a cut
**given that inflation is below 2%**.

We now split the historical record into two inflation regimes
and compute conditional probabilities for each Fed decision.

This is how a macro analyst actually uses probability —
not as a single number, but conditioned on the state of the world.

In [5]:
# =============================================================================
# CELL 8 — Conditional Probability: Fed Decisions Given Inflation Regime
# =============================================================================
# We pull CPI from FRED and classify each month as:
#   - High inflation: CPI YoY > 4%
#   - Low inflation:  CPI YoY <= 4%
#
# Then we compute P(Fed Decision | Inflation Regime)
# and compare to the unconditional probabilities.
# =============================================================================

# Pull CPI
print("Pulling CPI from FRED...")
cpi = get_fred_series("CPIAUCSL", FRED_KEY, from_date="2000-01-01")
print(f"  ✓ {len(cpi)} monthly observations loaded")

# Compute YoY inflation
cpi_yoy = cpi.pct_change(12).dropna()

# Align to month-end
cpi_yoy.index = cpi_yoy.index.to_period("M").to_timestamp("M")

# Classify inflation regime
high_inflation = cpi_yoy > 0.04
low_inflation  = cpi_yoy <= 0.04

# Align all series
common = decisions_monthly.index.intersection(cpi_yoy.index)
dec_   = decisions_monthly[common]
inf_   = cpi_yoy[common]
hi_    = high_inflation[common]
lo_    = low_inflation[common]

# ── CONDITIONAL PROBABILITIES ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("CONDITIONAL PROBABILITY — P(Fed Decision | Inflation Regime)")
print(f"{'='*65}")

print(f"\n  Inflation threshold: 4% YoY")
print(f"  High inflation months: {hi_.sum()} | Low inflation months: {lo_.sum()}")

print(f"\n  {'Decision':<8} {'P(D) unconditional':>20} "
      f"{'P(D|High Inf)':>15} {'P(D|Low Inf)':>14}")
print(f"  {'-'*60}")

for scenario in ["CUT", "HOLD", "HIKE"]:
    p_uncond   = (dec_ == scenario).mean()
    p_high_inf = (dec_[hi_] == scenario).mean()
    p_low_inf  = (dec_[lo_] == scenario).mean()
    print(f"  {scenario:<8} {p_uncond:>20.2%} {p_high_inf:>15.2%} {p_low_inf:>14.2%}")

print(f"  {'-'*60}")

# Key insight
p_cut_high = (dec_[hi_] == "CUT").mean()
p_cut_low  = (dec_[lo_] == "CUT").mean()
p_hike_high = (dec_[hi_] == "HIKE").mean()
p_hike_low  = (dec_[lo_] == "HIKE").mean()

print(f"""
  KEY INSIGHTS

  P(Cut | High Inflation) = {p_cut_high:.2%}
  P(Cut | Low Inflation)  = {p_cut_low:.2%}
  → The Fed cuts {p_cut_low/p_cut_high:.1f}x more often when inflation is low.

  P(Hike | High Inflation) = {p_hike_high:.2%}
  P(Hike | Low Inflation)  = {p_hike_low:.2%}
  → The Fed hikes {p_hike_high/p_hike_low:.1f}x more often when inflation is high.

  This is why conditioning matters.
  The unconditional probability of a cut ({(dec_=='CUT').mean():.2%}) is misleading
  without knowing the current inflation regime.

  CFA Formula verified:
  P(Cut ∩ High Inflation) = {(dec_[hi_]=='CUT').sum()}/{len(dec_)} = {(dec_[hi_]=='CUT').sum()/len(dec_):.2%}
  P(Cut | High Inflation) = {(dec_[hi_]=='CUT').sum()/len(dec_):.2%} / {hi_.mean():.2%} = {p_cut_high:.2%} ✓
""")
print(f"{'='*65}")

Pulling CPI from FRED...
  ✓ 289 monthly observations loaded

CONDITIONAL PROBABILITY — P(Fed Decision | Inflation Regime)

  Inflation threshold: 4% YoY
  High inflation months: 40 | Low inflation months: 237

  Decision   P(D) unconditional   P(D|High Inf)   P(D|Low Inf)
  ------------------------------------------------------------
  CUT                    25.99%          20.00%         27.00%
  HOLD                   34.66%          22.50%         36.71%
  HIKE                   39.35%          57.50%         36.29%
  ------------------------------------------------------------

  KEY INSIGHTS

  P(Cut | High Inflation) = 20.00%
  P(Cut | Low Inflation)  = 27.00%
  → The Fed cuts 1.4x more often when inflation is low.

  P(Hike | High Inflation) = 57.50%
  P(Hike | Low Inflation)  = 36.29%
  → The Fed hikes 1.6x more often when inflation is high.

  This is why conditioning matters.
  The unconditional probability of a cut (25.99%) is misleading
  without knowing the current inflat

## 4. Bayes' Theorem — Updating Your View When New Data Arrives

It is now the morning of the Fed meeting.

The CPI report just dropped: inflation came in at 3.2% — 
below the 4% threshold. Higher than expected.

Your PM asks:

> "Does this change our probability of a cut today?"

This is exactly what **Bayes' theorem** is for.

It lets you update a **prior probability** (what you believed before)
into a **posterior probability** (what you should believe now)
given new evidence.

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

In our context:
- $P(\text{Cut})$ — prior probability of a cut (your PM's view: 50%)
- $P(\text{Low CPI} | \text{Cut})$ — how often does low CPI precede a cut?
- $P(\text{Low CPI})$ — overall frequency of low CPI months
- $P(\text{Cut} | \text{Low CPI})$ — posterior: updated probability of a cut

We compute all of these from the historical FRED data.

In [7]:
# =============================================================================
# CELL 10 — Bayes' Theorem: Updating Fed Cut Probability on CPI Data
# =============================================================================
# Prior:    P(Cut) = 0.50  (PM's subjective view)
# Evidence: CPI came in below 4% (low inflation regime)
#
# We compute:
# P(Low CPI | Cut)  — likelihood
# P(Low CPI)        — marginal probability
# P(Cut | Low CPI)  — posterior via Bayes
# =============================================================================

# ── DEFINE EVENTS ─────────────────────────────────────────────────────────────
cut_mask     = dec_ == "CUT"
low_inf_mask = lo_

# Prior — PM's subjective view
p_cut_prior  = p_subjective["CUT"]  # 0.50

# Likelihood — P(Low CPI | Cut)
# Among all months where Fed cut, what fraction had low inflation?
p_low_given_cut = low_inf_mask[cut_mask].mean()

# Marginal — P(Low CPI)
p_low_inf = low_inf_mask.mean()

# Posterior — Bayes theorem
p_cut_posterior = (p_low_given_cut * p_cut_prior) / p_low_inf

# ── ALSO COMPUTE FOR HIKE ─────────────────────────────────────────────────────
hike_mask          = dec_ == "HIKE"
p_hike_prior       = p_subjective["HIKE"]
p_low_given_hike   = low_inf_mask[hike_mask].mean()
p_hike_posterior   = (p_low_given_hike * p_hike_prior) / p_low_inf

# Hold — derived (must sum to 1)
p_hold_posterior   = 1 - p_cut_posterior - p_hike_posterior

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"{'='*62}")
print("BAYES' THEOREM — Updating Fed Probabilities on CPI Data")
print(f"{'='*62}")

print(f"""
  EVIDENCE: CPI YoY = 3.2% → Low inflation regime confirmed

  BAYES FORMULA:
  P(Cut | Low CPI) = P(Low CPI | Cut) × P(Cut) / P(Low CPI)
""")

print(f"  {'Component':<30} {'Value':>10}")
print(f"  {'-'*42}")
print(f"  {'P(Cut) — prior':<30} {p_cut_prior:>10.2%}")
print(f"  {'P(Low CPI | Cut) — likelihood':<30} {p_low_given_cut:>10.2%}")
print(f"  {'P(Low CPI) — marginal':<30} {p_low_inf:>10.2%}")
print(f"  {'-'*42}")
print(f"  {'P(Cut | Low CPI) — posterior':<30} {p_cut_posterior:>10.2%}")

print(f"\n  {'='*42}")
print(f"  UPDATED PROBABILITIES AFTER CPI DATA")
print(f"  {'='*42}")
print(f"  {'Scenario':<10} {'Prior':>10} {'Posterior':>12} {'Change':>10}")
print(f"  {'-'*44}")
print(f"  {'CUT':<10} {p_cut_prior:>10.2%} {p_cut_posterior:>12.2%} "
      f"{p_cut_posterior-p_cut_prior:>+10.2%}")
print(f"  {'HOLD':<10} {p_subjective['HOLD']:>10.2%} {p_hold_posterior:>12.2%} "
      f"{p_hold_posterior-p_subjective['HOLD']:>+10.2%}")
print(f"  {'HIKE':<10} {p_hike_prior:>10.2%} {p_hike_posterior:>12.2%} "
      f"{p_hike_posterior-p_hike_prior:>+10.2%}")
print(f"  {'-'*44}")
print(f"  {'SUM':<10} {sum(p_subjective.values()):>10.2%} "
      f"{p_cut_posterior+p_hold_posterior+p_hike_posterior:>12.2%}")

print(f"""
  INTERPRETATION
  Before the CPI print, your PM assigned P(Cut) = {p_cut_prior:.0%}.
  After observing low inflation, Bayes updates this to {p_cut_posterior:.2%}.

  The CPI data {'increased' if p_cut_posterior > p_cut_prior else 'decreased'} 
  the probability of a cut by {abs(p_cut_posterior - p_cut_prior):.2%}.

  This is how a macro analyst updates their view in real time —
  not by gut feel, but by applying probability theory to new data.
""")
print(f"{'='*62}")

BAYES' THEOREM — Updating Fed Probabilities on CPI Data

  EVIDENCE: CPI YoY = 3.2% → Low inflation regime confirmed

  BAYES FORMULA:
  P(Cut | Low CPI) = P(Low CPI | Cut) × P(Cut) / P(Low CPI)

  Component                           Value
  ------------------------------------------
  P(Cut) — prior                     50.00%
  P(Low CPI | Cut) — likelihood      88.89%
  P(Low CPI) — marginal              85.56%
  ------------------------------------------
  P(Cut | Low CPI) — posterior       51.95%

  UPDATED PROBABILITIES AFTER CPI DATA
  Scenario        Prior    Posterior     Change
  --------------------------------------------
  CUT            50.00%       51.95%     +1.95%
  HOLD           40.00%       38.83%     -1.17%
  HIKE           10.00%        9.22%     -0.78%
  --------------------------------------------
  SUM           100.00%      100.00%

  INTERPRETATION
  Before the CPI print, your PM assigned P(Cut) = 50%.
  After observing low inflation, Bayes updates this to 51.

## 5. Portfolio Expected Return and Variance from Scratch

The Fed meeting is over. They held rates.

Now your PM asks the harder question:

> "What is our portfolio's expected return and risk
> going forward — and how does the correlation between
> SPY and TLT affect our overall volatility?"

This is the core of **portfolio mathematics** — CFA Reading 4.

For a two-asset portfolio:

$$E(R_p) = w_1 E(R_1) + w_2 E(R_2)$$

$$\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \sigma_1 \sigma_2 \rho_{1,2}$$

Where $\rho_{1,2}$ is the correlation between the two assets.

The correlation term is everything.
A negative correlation between SPY and TLT is what makes
the 60/40 portfolio work — in theory.
We will test whether it actually held during our sample period.

In [8]:
# =============================================================================
# CELL 12 — Portfolio Expected Return and Variance from Scratch
# =============================================================================
# We use the HOLD regime returns as our forward-looking assumption
# (Fed just held rates) and compute portfolio statistics.
# =============================================================================

# ── USE FULL SAMPLE FOR PORTFOLIO STATS ───────────────────────────────────────
# Align SPY and TLT monthly returns
common_pt = spy_monthly.index.intersection(tlt_monthly.index)
spy_m2    = spy_monthly[common_pt]
tlt_m2    = tlt_monthly[common_pt]

# ── INDIVIDUAL ASSET STATISTICS ───────────────────────────────────────────────
e_spy    = spy_m2.mean()
e_tlt    = tlt_m2.mean()
var_spy  = spy_m2.var(ddof=1)
var_tlt  = tlt_m2.var(ddof=1)
std_spy  = spy_m2.std(ddof=1)
std_tlt  = tlt_m2.std(ddof=1)

# Covariance and correlation
cov_matrix   = np.cov(spy_m2, tlt_m2, ddof=1)
cov_spy_tlt  = cov_matrix[0, 1]
corr_spy_tlt = cov_spy_tlt / (std_spy * std_tlt)

# ── PORTFOLIO STATISTICS ──────────────────────────────────────────────────────
w1 = w_spy  # 0.60
w2 = w_tlt  # 0.40

# Expected return
e_port = w1 * e_spy + w2 * e_tlt

# Variance — full formula
var_port = (w1**2 * var_spy +
            w2**2 * var_tlt +
            2 * w1 * w2 * cov_spy_tlt)

std_port = var_port ** 0.5

# ── ANNUALISE ─────────────────────────────────────────────────────────────────
e_port_ann   = e_port   * 12
std_port_ann = std_port * np.sqrt(12)
e_spy_ann    = e_spy    * 12
e_tlt_ann    = e_tlt    * 12
std_spy_ann  = std_spy  * np.sqrt(12)
std_tlt_ann  = std_tlt  * np.sqrt(12)

# ── DIVERSIFICATION BENEFIT ───────────────────────────────────────────────────
# Weighted average of individual vols vs portfolio vol
wtd_avg_vol  = w1 * std_spy_ann + w2 * std_tlt_ann
div_benefit  = wtd_avg_vol - std_port_ann

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"{'='*62}")
print("PORTFOLIO STATISTICS — 60% SPY + 40% TLT (2000–2023)")
print(f"{'='*62}")

print(f"\n  INDIVIDUAL ASSETS (Annualised)")
print(f"  {'Asset':<8} {'E(R)':>10} {'Volatility':>12} {'Sharpe (Rf=0)':>15}")
print(f"  {'-'*48}")
print(f"  {'SPY':<8} {e_spy_ann:>10.2%} {std_spy_ann:>12.2%} {e_spy_ann/std_spy_ann:>15.4f}")
print(f"  {'TLT':<8} {e_tlt_ann:>10.2%} {std_tlt_ann:>12.2%} {e_tlt_ann/std_tlt_ann:>15.4f}")

print(f"\n  CORRELATION & COVARIANCE")
print(f"  Covariance (monthly):     {cov_spy_tlt:.6f}")
print(f"  Correlation:              {corr_spy_tlt:.4f}")

if corr_spy_tlt < 0:
    print(f"  → Negative correlation ✓ — diversification benefit exists")
else:
    print(f"  → Positive correlation ✗ — limited diversification benefit")

print(f"\n  PORTFOLIO (Annualised)")
print(f"  {'Metric':<30} {'Value':>10}")
print(f"  {'-'*42}")
print(f"  {'Expected Return':<30} {e_port_ann:>10.2%}")
print(f"  {'Volatility':<30} {std_port_ann:>10.2%}")
print(f"  {'Sharpe Ratio (Rf=0)':<30} {e_port_ann/std_port_ann:>10.4f}")
print(f"  {'Weighted Avg Vol (no div)':<30} {wtd_avg_vol:>10.2%}")
print(f"  {'Diversification Benefit':<30} {div_benefit:>10.2%}")

print(f"""
  INTERPRETATION
  The correlation between SPY and TLT is {corr_spy_tlt:.4f}.
  This {'negative' if corr_spy_tlt < 0 else 'positive'} correlation reduces portfolio volatility
  from {wtd_avg_vol:.2%} (weighted average) to {std_port_ann:.2%} (actual).
  The diversification benefit is {div_benefit:.2%} per year.

  This is the mathematical foundation of the 60/40 portfolio.
  When correlation rises toward +1 (as it did in 2022),
  this benefit disappears — and both assets fall together.
""")
print(f"{'='*62}")

PORTFOLIO STATISTICS — 60% SPY + 40% TLT (2000–2023)

  INDIVIDUAL ASSETS (Annualised)
  Asset          E(R)   Volatility   Sharpe (Rf=0)
  ------------------------------------------------
  SPY           8.31%       14.99%          0.5542
  TLT           1.48%       13.72%          0.1077

  CORRELATION & COVARIANCE
  Covariance (monthly):     -0.000194
  Correlation:              -0.1131
  → Negative correlation ✓ — diversification benefit exists

  PORTFOLIO (Annualised)
  Metric                              Value
  ------------------------------------------
  Expected Return                     5.58%
  Volatility                          9.99%
  Sharpe Ratio (Rf=0)                0.5580
  Weighted Avg Vol (no div)          14.48%
  Diversification Benefit             4.49%

  INTERPRETATION
  The correlation between SPY and TLT is -0.1131.
  This negative correlation reduces portfolio volatility
  from 14.48% (weighted average) to 9.99% (actual).
  The diversification benefit is 4.

## 6. The 2022 Problem — When Correlation Breaks Down

The 60/40 portfolio lost approximately 16% in 2022.

Both SPY and TLT fell simultaneously.
The negative correlation that justified the allocation disappeared.

This is not a black swan — it is a known regime risk.
When inflation is high, bonds and equities become positively correlated
because rising rates hurt both asset classes.

We now split the sample into two regimes:
- **Normal regime**: inflation below 4%
- **High inflation regime**: inflation above 4%

And compute the correlation between SPY and TLT in each.

This is the practical application of conditional probability
to portfolio risk management — exactly what a risk manager
at a fund would do after 2022.

In [9]:
# =============================================================================
# CELL 14 — Correlation Breakdown: Normal vs High Inflation Regime
# =============================================================================

# Align CPI YoY with monthly returns
common_all = spy_monthly.index.intersection(tlt_monthly.index).intersection(cpi_yoy.index)
spy_r      = spy_monthly[common_all]
tlt_r      = tlt_monthly[common_all]
inf_regime = cpi_yoy[common_all]

high_inf_mask = inf_regime > 0.04
low_inf_mask  = inf_regime <= 0.04

# ── CORRELATION BY REGIME ─────────────────────────────────────────────────────
corr_full = spy_r.corr(tlt_r)
corr_low  = spy_r[low_inf_mask].corr(tlt_r[low_inf_mask])
corr_high = spy_r[high_inf_mask].corr(tlt_r[high_inf_mask])

# ── PORTFOLIO VOL BY REGIME ───────────────────────────────────────────────────
def portfolio_vol(spy_ret, tlt_ret, w1=0.60, w2=0.40):
    std1   = spy_ret.std(ddof=1) * np.sqrt(12)
    std2   = tlt_ret.std(ddof=1) * np.sqrt(12)
    corr   = spy_ret.corr(tlt_ret)
    cov    = corr * std1 * std2
    var_p  = w1**2 * std1**2 + w2**2 * std2**2 + 2 * w1 * w2 * cov
    return var_p ** 0.5

vol_full = portfolio_vol(spy_r, tlt_r)
vol_low  = portfolio_vol(spy_r[low_inf_mask], tlt_r[low_inf_mask])
vol_high = portfolio_vol(spy_r[high_inf_mask], tlt_r[high_inf_mask])

# ── EXPECTED RETURN BY REGIME ─────────────────────────────────────────────────
e_port_low  = (0.60 * spy_r[low_inf_mask].mean()  + 0.40 * tlt_r[low_inf_mask].mean())  * 12
e_port_high = (0.60 * spy_r[high_inf_mask].mean() + 0.40 * tlt_r[high_inf_mask].mean()) * 12

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"{'='*62}")
print("CORRELATION BREAKDOWN BY INFLATION REGIME — 60/40 Portfolio")
print(f"{'='*62}")

print(f"\n  {'Regime':<25} {'Months':>7} {'Correlation':>13} "
      f"{'Port Vol':>10} {'Port E(R)':>11}")
print(f"  {'-'*68}")
print(f"  {'Full sample':<25} {len(spy_r):>7} {corr_full:>13.4f} "
      f"{vol_full:>10.2%} {e_port_ann:>11.2%}")
print(f"  {'Low inflation (≤4%)':<25} {low_inf_mask.sum():>7} {corr_low:>13.4f} "
      f"{vol_low:>10.2%} {e_port_low:>11.2%}")
print(f"  {'High inflation (>4%)':<25} {high_inf_mask.sum():>7} {corr_high:>13.4f} "
      f"{vol_high:>10.2%} {e_port_high:>11.2%}")

print(f"""
  KEY FINDINGS

  In low inflation regimes:
  → Correlation = {corr_low:.4f} (negative — diversification works)
  → Portfolio volatility = {vol_low:.2%}
  → Expected return      = {e_port_low:.2%}

  In high inflation regimes:
  → Correlation = {corr_high:.4f} ({'positive' if corr_high > 0 else 'negative'} — diversification {'breaks down' if corr_high > 0 else 'holds'})
  → Portfolio volatility = {vol_high:.2%}
  → Expected return      = {e_port_high:.2%}

  The correlation shift from {corr_low:.4f} to {corr_high:.4f}
  increases portfolio volatility by {vol_high - vol_low:.2%} per year.

  This is exactly what happened in 2022.
  The 60/40 portfolio is not broken — it is regime-dependent.
  Understanding when it works and when it doesn't
  is what separates a quant from a textbook reader.
""")
print(f"{'='*62}")

CORRELATION BREAKDOWN BY INFLATION REGIME — 60/40 Portfolio

  Regime                     Months   Correlation   Port Vol   Port E(R)
  --------------------------------------------------------------------
  Full sample                   238       -0.1131      9.99%       5.58%
  Low inflation (≤4%)           198       -0.2419      9.04%       8.04%
  High inflation (>4%)           40        0.3756     13.37%      -6.60%

  KEY FINDINGS

  In low inflation regimes:
  → Correlation = -0.2419 (negative — diversification works)
  → Portfolio volatility = 9.04%
  → Expected return      = 8.04%

  In high inflation regimes:
  → Correlation = 0.3756 (positive — diversification breaks down)
  → Portfolio volatility = 13.37%
  → Expected return      = -6.60%

  The correlation shift from -0.2419 to 0.3756
  increases portfolio volatility by 4.32% per year.

  This is exactly what happened in 2022.
  The 60/40 portfolio is not broken — it is regime-dependent.
  Understanding when it works and wh

## 7. CFA Exam Style Practice Problems

Three problems written in CFA exam style.

All numbers are derived from the real data in this notebook.

Try each one before reading the solution.

In [11]:
# =============================================================================
# CELL 16 — CFA Exam Style Practice Problems
# =============================================================================

print("=" * 65)
print("PRACTICE PROBLEMS — Probability Concepts")
print("=" * 65)

# ── PROBLEM 1 ─────────────────────────────────────────────────────────────────
print("""
PROBLEM 1
─────────────────────────────────────────────────────────────
Based on historical FRED data (2000–2023), the following
probabilities have been estimated for Fed decisions:

  P(Cut)  = 26%
  P(Hold) = 35%
  P(Hike) = 39%

Additionally:
  P(Low Inflation | Cut)  = 89%
  P(Low Inflation | Hold) = 88%
  P(Low Inflation | Hike) = 83%

A) Calculate P(Low Inflation) using the total probability rule.
B) Given that inflation is low, calculate P(Cut | Low Inflation)
   using Bayes' theorem.
─────────────────────────────────────────────────────────────""")

# Answers
p_cut_h  = 0.26
p_hold_h = 0.35
p_hike_h = 0.39

p_low_cut  = p_low_given_cut
p_low_hold = (dec_[dec_=="HOLD"].index.isin(low_inf_mask[low_inf_mask].index)).mean()
p_low_hike = low_inf_mask[hike_mask].mean()

# Total probability rule
p_low_total = (p_cut_h  * p_low_cut +
               p_hold_h * p_low_hold +
               p_hike_h * p_low_hike)

# Bayes
p_cut_given_low = (p_low_cut * p_cut_h) / p_low_total

print("ANSWER:")
print(f"  A) P(Low Inf) = P(Low|Cut)×P(Cut) + P(Low|Hold)×P(Hold) + P(Low|Hike)×P(Hike)")
print(f"               = {p_low_cut:.2%}×{p_cut_h:.0%} + {p_low_hold:.2%}×{p_hold_h:.0%} + {p_low_hike:.2%}×{p_hike_h:.0%}")
print(f"               = {p_low_total:.4%}")
print(f"\n  B) P(Cut|Low Inf) = P(Low Inf|Cut) × P(Cut) / P(Low Inf)")
print(f"                    = {p_low_cut:.2%} × {p_cut_h:.0%} / {p_low_total:.4%}")
print(f"                    = {p_cut_given_low:.4%}")

# ── PROBLEM 2 ─────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("""
PROBLEM 2
─────────────────────────────────────────────────────────────
A portfolio consists of 60% SPY and 40% TLT.
Based on data from 2000–2023:

  E(R_SPY) = 8.31% per year
  E(R_TLT) = 1.48% per year
  σ(SPY)   = 14.99% per year
  σ(TLT)   = 13.72% per year
  ρ(SPY,TLT) = −0.1131

A) Calculate the portfolio expected return.
B) Calculate the portfolio variance and standard deviation.
C) What is the diversification benefit vs a weighted average
   of individual volatilities?
─────────────────────────────────────────────────────────────""")

e1, e2   = e_spy_ann, e_tlt_ann
s1, s2   = std_spy_ann, std_tlt_ann
rho      = corr_spy_tlt
w1, w2   = 0.60, 0.40

ep   = w1*e1 + w2*e2
varp = w1**2*s1**2 + w2**2*s2**2 + 2*w1*w2*rho*s1*s2
stdp = varp**0.5
wtd  = w1*s1 + w2*s2

print("ANSWER:")
print(f"  A) E(Rp) = 0.60×{e1:.2%} + 0.40×{e2:.2%} = {ep:.2%}")
print(f"\n  B) Var(p) = (0.60²×{s1:.2%}²) + (0.40²×{s2:.2%}²)")
print(f"             + 2×0.60×0.40×{rho:.4f}×{s1:.2%}×{s2:.2%}")
print(f"           = {varp:.6f}")
print(f"     σ(p)  = √{varp:.6f} = {stdp:.2%}")
print(f"\n  C) Weighted avg vol = 0.60×{s1:.2%} + 0.40×{s2:.2%} = {wtd:.2%}")
print(f"     Portfolio vol    = {stdp:.2%}")
print(f"     Diversification benefit = {wtd-stdp:.2%}")

# ── PROBLEM 3 ─────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("""
PROBLEM 3
─────────────────────────────────────────────────────────────
An analyst states:
"The 60/40 portfolio always benefits from diversification
because bonds and equities are negatively correlated."

Using the data from this notebook, evaluate this statement.
Is it always true? Under what conditions does it break down?
─────────────────────────────────────────────────────────────""")

print("ANSWER:")
print(f"  The statement is FALSE as an unconditional claim.")
print(f"\n  Full sample correlation:          {corr_full:.4f} (negative ✓)")
print(f"  Low inflation correlation:        {corr_low:.4f} (negative ✓)")
print(f"  High inflation correlation:       {corr_high:.4f} (positive ✗)")
print(f"\n  In high inflation regimes (>{4}% CPI YoY):")
print(f"  → Correlation turns positive ({corr_high:.4f})")
print(f"  → Portfolio volatility rises from {vol_low:.2%} to {vol_high:.2%}")
print(f"  → Expected return falls from {e_port_low:.2%} to {e_port_high:.2%}")
print(f"\n  The negative correlation is regime-dependent, not structural.")
print(f"  When both assets are driven by the same factor (rising rates),")
print(f"  diversification breaks down — as it did in 2022.")
print(f"{'='*65}")

PRACTICE PROBLEMS — Probability Concepts

PROBLEM 1
─────────────────────────────────────────────────────────────
Based on historical FRED data (2000–2023), the following
probabilities have been estimated for Fed decisions:

  P(Cut)  = 26%
  P(Hold) = 35%
  P(Hike) = 39%

Additionally:
  P(Low Inflation | Cut)  = 89%
  P(Low Inflation | Hold) = 88%
  P(Low Inflation | Hike) = 83%

A) Calculate P(Low Inflation) using the total probability rule.
B) Given that inflation is low, calculate P(Cut | Low Inflation)
   using Bayes' theorem.
─────────────────────────────────────────────────────────────
ANSWER:
  A) P(Low Inf) = P(Low|Cut)×P(Cut) + P(Low|Hold)×P(Hold) + P(Low|Hike)×P(Hike)
               = 88.89%×26% + 83.33%×35% + 77.00%×39%
               = 82.3078%

  B) P(Cut|Low Inf) = P(Low Inf|Cut) × P(Cut) / P(Low Inf)
                    = 88.89% × 26% / 82.3078%
                    = 28.0789%


PROBLEM 2
─────────────────────────────────────────────────────────────
A portfolio consists

## 8. Key Takeaways

---

### What you built in this notebook

You started with a PM asking "what is the probability the Fed cuts?"
and ended with a full probabilistic framework for macro investing.

| Concept | Applied To |
|---------|-----------|
| Exhaustive & mutually exclusive events | Fed decision scenarios |
| Empirical vs subjective probability | Historical FRED data vs PM view |
| Conditional probability | Fed decisions given inflation regime |
| Bayes' theorem | Updating cut probability on CPI print |
| Expected value | Portfolio return under macro scenarios |
| Portfolio variance | 60/40 SPY/TLT from scratch |
| Correlation & diversification | Regime-dependent breakdown in 2022 |

---

### The three things to remember

**1. Always condition your probabilities.**
P(Cut) = 26% unconditionally.
P(Cut | Low Inflation) = 27%. P(Cut | High Inflation) = 20%.
The unconditional number is rarely the right one to use.

**2. Bayes' theorem is how you update, not guess.**
When new data arrives, you don't throw away your prior.
You update it systematically. That is what separates
a rigorous analyst from one who reacts to headlines.

**3. Correlation is not a constant.**
The entire case for the 60/40 portfolio rests on
a negative SPY/TLT correlation.
In high inflation regimes that correlation turns positive
and the diversification benefit disappears entirely.
Models built on full-sample correlations will fail
exactly when you need them most.

---

### What comes next

| Notebook | Topic |
|----------|-------|
| ✅ QM 01 | Rates and Returns |
| ✅ QM 02 | Time Value of Money |
| ✅ QM 03 | Statistical Measures of Asset Returns |
| ✅ QM 04 | Probability Concepts |
| 🔜 QM 05 | Common Probability Distributions |
| 🔜 QM 06 | Sampling & Estimation |
| 🔜 QM 07 | Hypothesis Testing |
| 🔜 QM 08 | Introduction to Linear Regression |

---

*All notebooks are free. Always.*